# ADK Method 3 - Programmatic Execution in Google Colab

Course context: Google Skills, Build Your First Agent with Agent Development Kit (ADK), section **Other ways to run your agent - Three deployment methods**.

This notebook records the third method: running an ADK agent directly from Python code. It is useful for Colab, notebooks, data pipelines, and custom educational applications.

SecuredMe Education angle: this is the bridge between a lesson notebook and a future companion side panel. Instead of relying only on `adk web` or a terminal, a Python application can create a session, send a structured message, and collect an agent response.

## Safety Boundary

- Do not paste API keys into notebook cells.
- In Colab, add a secret named `GOOGLE_API_KEY` through the Secrets panel.
- This notebook uses an in-memory session service only.
- This is a learning notebook, not a production deployment.
- No learner data, private project data, or personal research files are used.

In [ ]:
!pip -q install google-adk

In [ ]:
import os

try:
    from google.colab import userdata
    api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    api_key = os.environ.get("GOOGLE_API_KEY")

if not api_key:
    raise RuntimeError(
        "Set GOOGLE_API_KEY in Colab Secrets before running this notebook."
    )

os.environ["GOOGLE_API_KEY"] = api_key
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"

print("GOOGLE_API_KEY loaded from secret/environment. The value is not displayed.")

## Define The Agent

This mirrors the local ADK learning agent but keeps the example self-contained for Colab. The model is set to `gemini-3.1-flash-lite` because the local session observed repeated temporary `503 UNAVAILABLE` spikes on heavier models.

In [ ]:
from google.adk.agents import Agent

agent = Agent(
    model="gemini-3.1-flash-lite",
    name="math_tutor_agent",
    description=(
        "Helps learners understand mathematics, with particular strength in "
        "fractal geometry, neutrosophic mathematics, and applied plithogeny."
    ),
    instruction=(
        "You are a patient mathematics tutor for learners at any level. "
        "Teach step by step, define every symbol before using it, and adapt "
        "the depth of the explanation to the learner's question. Use short "
        "worked examples when they help. If a question lacks enough "
        "information, ask for the missing detail instead of guessing. End "
        "each explanation with one brief comprehension question or practice step."
    ),
)

agent

## Create A Session And Runner

The session is managed in code. That is the core lesson of method 3: the application controls the session lifecycle instead of relying only on a terminal or web UI.

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

APP_NAME = "securedme_math_tutor_colab"
USER_ID = "student_1"
SESSION_ID = "session_001"

session_service = InMemorySessionService()
runner = Runner(
    agent=agent,
    app_name=APP_NAME,
    session_service=session_service,
)

print("Runner ready for", APP_NAME)

In [ ]:
async def run_agent(question: str) -> str:
    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID,
    )

    user_message = Content(
        role="user",
        parts=[Part(text=question)],
    )

    final_text = ""
    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=user_message,
    ):
        if event.is_final_response() and event.content and event.content.parts:
            final_text = event.content.parts[0].text

    return final_text

In [ ]:
question = "How do I solve 2x + 5 = 13?"
answer = await run_agent(question)

print("User:", question)
print("Agent:\n", answer)

## What This Means For SecuredMe Education

Programmatic execution is the most relevant method for notebooks, learning experiments, and controlled educational workflows. A future side panel or game loop can follow the same shape: create or resume a session, send a bounded learner message, collect the response, and decide what the learner sees next.

This notebook does not implement production security, persistent storage, CCP, context caching, tools, or multi-agent orchestration. It only proves the application-level control pattern.